# ▶ Run This Before The Session Starts

> Run all cells in this section **before the audience arrives**. This downloads and loads the models (~3–5 minutes). Once both ✅ messages appear, also pre-run the UMAP cell in Part 3, then collapse this section.


In [ ]:
# Install required packages (umap-learn not pre-installed on Colab)
!pip install -q umap-learn==0.5.6
print('✅ Packages ready')

In [ ]:
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import matplotlib
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from IPython.display import display, HTML
from transformers import BertTokenizer
import gensim.downloader as api

print('✅ Imports ready')

In [ ]:
print('Loading BERT tokenizer...')
tokenizer = BertTokenizer.from_pretrained('bert-base-uncased')
print('✅ BERT tokenizer loaded')

In [ ]:
print('Loading word2vec-google-news-300 (~1.6 GB, takes 2–3 minutes)...')
w2v = api.load('word2vec-google-news-300')
print(f'✅ word2vec loaded: {len(w2v):,} words')

---
# 📖 PART 1 — Tokenization

> **BERT uses BPE (Byte-Pair Encoding).** The same surface word can map to **different token IDs** depending on how it appears in a phrase. This is why context matters from the very first step of processing.


In [ ]:
def render_tokens(text, tokenizer):
    """Render BERT tokens as styled HTML chip cards with token IDs."""
    tokens = tokenizer.tokenize(text)
    ids = tokenizer.convert_tokens_to_ids(tokens)

    chips_html = "".join([
        f"""
        <div style="display:inline-block; margin:6px; text-align:center;">
          <div style="
            background:#0f172a;
            border:2px solid #22d3ee;
            border-radius:20px;
            padding:8px 16px;
            color:#f1f5f9;
            font-family:monospace;
            font-size:15px;
            font-weight:600;
          ">{tok}</div>
          <div style="
            color:#64748b;
            font-family:monospace;
            font-size:11px;
            margin-top:4px;
          ">ID: {id_}</div>
        </div>"""
        for tok, id_ in zip(tokens, ids)
    ])

    html = f"""
    <div style="
      background:#1e293b;
      border-radius:12px;
      padding:20px 24px;
      margin:12px 0;
      font-family:monospace;
    ">
      <div style="color:#94a3b8; font-size:12px; margin-bottom:12px;">INPUT TEXT</div>
      <div style="color:#e2e8f0; font-size:18px; font-weight:700; margin-bottom:16px;">&quot;{text}&quot;</div>
      <div style="color:#94a3b8; font-size:12px; margin-bottom:8px;">TOKENS ({len(tokens)} total)</div>
      <div>{chips_html}</div>
    </div>
    """
    return HTML(html)

print('✅ Helper function ready')

In [ ]:
display(render_tokens('river bank', tokenizer))

In [ ]:
display(render_tokens('bank account', tokenizer))

In [ ]:
# Side-by-side comparison
phrases = ['river bank', 'bank account']
results = []
for phrase in phrases:
    tokens = tokenizer.tokenize(phrase)
    ids = tokenizer.convert_tokens_to_ids(tokens)
    results.append(list(zip(tokens, ids)))

# Find the token ID for 'bank' in each phrase
bank_ids = []
for pairs in results:
    for tok, id_ in pairs:
        if tok == 'bank':
            bank_ids.append(id_)
            break

def row_html(pairs, highlight_token='bank'):
    cells_html = ''.join([
        f'<td style="padding:6px 12px; font-family:monospace; '
        f'color:{"#e879f9" if tok==highlight_token else "#e2e8f0"}; '
        f'font-weight:{"700" if tok==highlight_token else "400"};'>'
        f'{tok}<br><span style="color:#64748b; font-size:11px;">ID: {id_}</span></td>'
        for tok, id_ in pairs
    ])
    return f'<tr>{cells_html}</tr>'

html = f"""
<div style='background:#1e293b; border-radius:12px; padding:20px 24px; margin:12px 0;'>
  <div style='color:#94a3b8; font-size:12px; margin-bottom:12px;'>
    SIDE-BY-SIDE — <span style='color:#e879f9;'>bank</span> gets a different ID in each phrase
  </div>
  <table style='border-collapse:collapse; width:100%;'>
    <thead>
      <tr>
        <th colspan='{len(results[0])}' style='text-align:left; padding:4px 12px; color:#22d3ee; font-family:monospace; font-size:13px; border-bottom:1px solid #334155;'>&quot;river bank&quot;</th>
        <th colspan='{len(results[1])}' style='text-align:left; padding:4px 12px; color:#22d3ee; font-family:monospace; font-size:13px; border-bottom:1px solid #334155;'>&quot;bank account&quot;</th>
      </tr>
    </thead>
    <tbody>
      {row_html(results[0])}
      {row_html(results[1])}
    </tbody>
  </table>
  <div style='margin-top:16px; color:#94a3b8; font-size:13px;'>
    <span style='color:#e879f9; font-weight:700;'>bank</span>
    → ID <strong style='color:#f1f5f9;'>{bank_ids[0]}</strong> in &quot;river bank&quot;
    &nbsp;|&nbsp;
    ID <strong style='color:#f1f5f9;'>{bank_ids[1]}</strong> in &quot;bank account&quot;
  </div>
</div>
"""
display(HTML(html))

---
# 📐 PART 2 — Word Similarity

> **Words with similar meaning end up close in vector space.** word2vec learned this purely from co-occurrence patterns across billions of text tokens — no human labeling required.


In [ ]:
# ✏️  Change this word live — invite the audience to suggest one!
query_word = "king"

results = w2v.most_similar(query_word, topn=10)

rows = "".join([
    f"""
    <tr>
      <td style="padding:6px 16px; color:#94a3b8; font-family:monospace;">#{i+1}</td>
      <td style="padding:6px 16px; color:#e2e8f0; font-family:monospace; font-weight:600;">{word}</td>
      <td style="padding:6px 16px; color:#22d3ee; font-family:monospace;">{score:.4f}</td>
    </tr>"""
    for i, (word, score) in enumerate(results)
])

html = f"""
<div style="background:#1e293b; border-radius:12px; padding:20px 24px; max-width:460px;">
  <div style="color:#94a3b8; font-size:12px; margin-bottom:4px;">MOST SIMILAR TO</div>
  <div style="color:#f1f5f9; font-size:24px; font-weight:700; margin-bottom:16px; font-family:monospace;">&quot;{query_word}&quot;</div>
  <table style="border-collapse:collapse; width:100%;">
    <thead>
      <tr style="border-bottom:1px solid #334155;">
        <th style="padding:4px 16px; color:#64748b; text-align:left; font-size:11px;">#</th>
        <th style="padding:4px 16px; color:#64748b; text-align:left; font-size:11px;">WORD</th>
        <th style="padding:4px 16px; color:#64748b; text-align:left; font-size:11px;">SIMILARITY</th>
      </tr>
    </thead>
    <tbody>{rows}</tbody>
  </table>
</div>
"""
display(HTML(html))

### Arithmetic on Meaning

> If meaning is geometry, we can **add and subtract** concepts.
>
> The famous example: &nbsp; **king − man + woman = ?**


In [ ]:
results = w2v.most_similar(positive=["king", "woman"], negative=["man"], topn=5)

rows = "".join([
    f"""
    <tr style="background:{"#1e1040" if word.lower()=="queen" else "transparent"}">
      <td style="padding:8px 16px; font-size:20px;">{"👑" if word.lower()=="queen" else ""}</td>
      <td style="padding:8px 16px; color:{"#e879f9" if word.lower()=='queen' else '#e2e8f0'}; font-family:monospace; font-weight:{"700" if word.lower()=='queen' else '400'}; font-size:16px;">{word}</td>
      <td style="padding:8px 16px; color:{"#e879f9" if word.lower()=='queen' else '#22d3ee'}; font-family:monospace;">{score:.4f}</td>
    </tr>"""
    for word, score in results
])

html = f"""
<div style="background:#1e293b; border-radius:12px; padding:20px 24px; max-width:460px;">
  <div style="color:#f1f5f9; font-size:18px; font-weight:700; margin-bottom:4px; font-family:monospace;">
    king &minus; man + woman = ?
  </div>
  <div style="color:#64748b; font-size:12px; margin-bottom:16px;">Top results — word2vec-google-news-300</div>
  <table style="border-collapse:collapse; width:100%;">
    <tbody>{rows}</tbody>
  </table>
</div>
"""
display(HTML(html))

---
# 🗺️ PART 3 — Embedding Map

> **Let's see the whole neighborhood at once.** UMAP compresses 300-dimensional word vectors into 2D so we can visualize the semantic clusters.

> ⚠️ **Presenter note:** Run this cell during SETUP and **do not re-run it live** — UMAP is non-deterministic and the layout will shift between runs.


In [ ]:
import umap

# Curated word list — 5 semantic categories
categories = {
    'Royalty':     {'words': ['king', 'queen', 'prince', 'princess', 'throne', 'crown'],             'color': '#f59e0b'},
    'Animals':     {'words': ['dog', 'cat', 'wolf', 'horse', 'lion', 'tiger'],                       'color': '#2dd4bf'},
    'Geography':   {'words': ['river', 'ocean', 'mountain', 'forest', 'valley', 'beach'],            'color': '#4ade80'},
    'Professions': {'words': ['doctor', 'nurse', 'engineer', 'teacher', 'lawyer', 'scientist'],      'color': '#a78bfa'},
    'Finance':     {'words': ['money', 'loan', 'credit', 'investment', 'bank', 'currency'],          'color': '#fb923c'},
}

all_words, all_colors, all_labels = [], [], []
for cat_name, cat in categories.items():
    for word in cat['words']:
        if word in w2v:
            all_words.append(word)
            all_colors.append(cat['color'])
            all_labels.append(cat_name)

vectors = np.array([w2v[w] for w in all_words])

reducer = umap.UMAP(n_components=2, random_state=42, n_neighbors=8, min_dist=0.3)
embedding = reducer.fit_transform(vectors)

fig, ax = plt.subplots(figsize=(13, 8))
fig.patch.set_facecolor('#0f172a')
ax.set_facecolor('#0f172a')

ax.scatter(embedding[:, 0], embedding[:, 1], c=all_colors, s=130, alpha=0.92, zorder=3)

for i, word in enumerate(all_words):
    is_bank = word == 'bank'
    ax.annotate(
        word,
        (embedding[i, 0], embedding[i, 1]),
        xytext=(7, 7), textcoords='offset points',
        fontsize=11,
        color='#e879f9' if is_bank else '#e2e8f0',
        fontfamily='monospace',
        fontweight='bold' if is_bank else 'normal',
    )

legend_patches = [
    mpatches.Patch(color=cat['color'], label=cat_name)
    for cat_name, cat in categories.items()
]
ax.legend(
    handles=legend_patches, loc='lower right',
    facecolor='#1e293b', edgecolor='#334155',
    labelcolor='#e2e8f0', fontsize=11
)

ax.set_xticks([])
ax.set_yticks([])
for spine in ax.spines.values():
    spine.set_edgecolor('#334155')

ax.set_title(
    'Word Embedding Space — UMAP 2D Projection',
    color='#94a3b8', fontsize=13, pad=16, fontfamily='monospace'
)

plt.tight_layout()
plt.show()

---
# 🔁 Closing — Back to the Puzzle

> We started with one question:
>
> *Why does the word **bank** seem simple to us, but difficult for a model?*
>
> Let's look at the evidence we collected.


In [ ]:
# Same word — two contexts — two different token representations
display(render_tokens('river bank', tokenizer))
display(render_tokens('bank account', tokenizer))

---
## Why is *bank* difficult for a model?

| Layer | What we saw |
|---|---|
| **Tokenization** | Same surface word → different token IDs depending on phrase context |
| **Embeddings** | *bank* sits near *money* and *loan* in the Finance cluster |
| **Context** | Static models assign one fixed vector; BERT reads meaning from sentence context |

> **Same surface. Different meaning. Different representation.**
>
> *Modern NLP is representation, geometry, and context at scale.*
